# Feature Construction for better Predictions

This Notebook explores a set of feature engineering and construction approaches which aims to enhance prediction score on the Titanic dataset. Our main goal here is to enhance and clean the data available to us in such a way that it improves covariance between our independent features and target Variable **Survived**.

In short, this Notebook does the following:

1. Loading the dataset
2. Initial feature covariance analysis
3. Handling null values
4. Feature engineering
5. Final covariance matrix check
6. Dataset preperation for fitting in models
7. Model evaluation
8. Final submission 

**Please Note: I wrote this notebook almost concurrently with my colleague Ankit1743, who proposed and showed an Alternate approach for Feature engineering in his Notebook which can ne viewed here:**
https://www.kaggle.com/code/ankit1743/improve-score-on-titanic-by-feature-engineering

### 1. Loading the dataset

Firstly, we load the datasets in Focus, namely the Train and Test datasets using Pandas library's 'read_csv' function.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import re
import warnings
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
traindf=pd.read_csv("/kaggle/input/titanic/train.csv")
testdf=pd.read_csv("/kaggle/input/titanic/test.csv")

### 2. Initial feature covariance analysis

Let's perform an initial analysis of covariance between our Features and Target Feature/Variable. for this two major steps have been done prior to plotting the Covariance matrix using Seaborn library:
1. Label Encoding using Sci-kit's **LabeEncoder function**, in order to convert all data into numeric for evaluation. It is important since Seaborn's covariance matrix **does'nt** accept categorical values.
2. Scaling has been done usin Sci-Kit's **StandardScaler function** to Standardize the scale of data such that it fits standard Normal distribution. This helps us in getting a standardized covariance score **between -1 and +1**.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
encoder = LabelEncoder()
scaler= StandardScaler()
tren = traindf.apply(lambda i: encoder.fit_transform(i) if i.dtype == 'O' else i)
trcov = pd.DataFrame(scaler.fit_transform(tren), columns=tren.columns)
cov_matrix = trcov.cov()
plt.figure(figsize=(12, 12))
sns.heatmap(cov_matrix, annot=True, fmt='.3f', linewidths=.75)
plt.title('Covariance')
plt.show()

### 3. Handling null values

To handle null values, I have created a function **numnull** which will yield number of null values in each column to help us make a pre-requisite analysis on our columns of focus.

In [ ]:
def numnull(df):
    for i in df:
        col=i
        ct= df[col].value_counts(dropna=False).sort_index()
        nan=df[col].isna().sum() 
        print("Number of null values in ",col, "is: ", nan)

In [ ]:
numnull(traindf)

In [ ]:
numnull(testdf)

As we can see, 'Fare' and 'Embarked' have bare minimum null values in both datasets, this can be handled simply by replacing it wIth **mode** value in Embarked and and **mean** value in Fare.

In [ ]:
traindf['Embarked'].fillna(traindf['Embarked'].mode()[0], inplace=True)
testdf['Fare'].fillna(testdf['Fare'].mean(), inplace=True)

Features 'Age' and 'Cabin' have lot of null values to handle. we will do this ask as we further analyze our data while performing feature engineering.

### 4. Feature engineering

Now comes the most important part, the Feature Engineering. here we will visualize some Features using the **insights function** I made, and try to figure out how we can clean and improve their values for better predicitons.

The insifgts function makes a **stacked bar plot** of 'Feature 1' and 'Feature 2', allowing us to compare how each value of 'Feature 1' categorizes into values of 'feature 2'. 

In [ ]:
import matplotlib.pyplot as plt
def insights(data, cn1, cn2):
    ct= data[cn1].value_counts(dropna=False).sort_index()
    grouped_data = data.groupby([cn1, cn2]).size().unstack(fill_value=0)
    proportions = grouped_data.div(grouped_data.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(10, 10))
    proportions.plot(kind='bar', stacked=True, ax=ax)
    plt.xlabel(cn1)
    plt.ylabel('Ratio')
    plt.legend(title=cn2, bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
insights(traindf, 'Age', 'Survived')

This plot shows that there does exists a pattern in ages of people who Survived. While most of the infants did survive, a significant portion of kids beween the age 1 and 11 unfortunately perished. A small yet significant spike is observed between the ages 11 and 18, but after that, especially after the age of 50, most people perished. 

Clearly we can convert these raw ages into categories as follow:

In [ ]:
def group_age(val):
    if val <1:
        return 0
    elif 1<= val <11:
        return 1
    elif 11<= val <18:
        return 2
    elif 18<= val <52:
        return 3
    else:
        return 4

traindf['Age'] = traindf['Age'].apply(group_age)
testdf['Age'] = testdf['Age'].apply(group_age)

The **group_age function** groups ages into distinct categories, which can be translated as follow:-
1. 0-> infants
2. 1-> kids
3. 2-> adoloscents
4. 3-> adults
6. 4-> senior citizens

In [ ]:
insights(traindf, 'Age', 'Survived')

Unlike the sister notebook, here we will make a new feature **RelativeNum**, which tells the total number of relative by summing the Parch and SibSp columns, hoping to get better results.

In [ ]:
traindf['RelativeNum'] = traindf['SibSp'] + traindf['Parch']
testdf['RelativeNum'] = testdf['SibSp'] + testdf['Parch']

In [ ]:
insights(traindf, 'RelativeNum', 'Survived')

Clearly, we can notice that Total number of relatives play a big part in understanding who survived. As we see, most of solo travelers perished, travellers with nuclear families had the highest chance of survival, while chances of survival decrease subsequently as number of relatives becomes larger.

In [ ]:
def group_rel(val):
    if val <1:
        return 0
    elif 1<= val <4:
        return 1
    elif 4<= val <7:
        return 2
    else:
        return 3
traindf['RelativeNum'] = traindf['RelativeNum'].apply(group_rel)
testdf['RelativeNum'] = testdf['RelativeNum'].apply(group_rel)

The **group_rel function** separates number of relatives into distinct categories, which can be translated as follow:-
1. 0-> Solo travellers
2. 1-> Nuclear families
3. 2-> Large families
4. 3-> Group travellers

In [ ]:
insights(traindf, 'RelativeNum', 'Survived')

After grouping the Numrelatives feature, we obtain a much more meaningful and easy to understand data.

Now we will try to use this newly constructed feature for predictiong the missing values of Age feature using Insight function.

In [ ]:
insights(traindf, 'RelativeNum', 'Age')

As we see above, most of people travelling solo, or with a partner, were between the age group 18-52, whereas most of the ones travelling in small families were kids below 11. Furthermore, a majority of travellers travelling in large groups were senior citizens. This gives us a solid base to predict the missing values in Age feature, as done below:

In [ ]:
def pos_age(row):
    if pd.isnull(row['Age']):
        if row['RelativeNum'] <2:
            return 3
        elif row['RelativeNum'] ==2 :
            return 1
        elif row['RelativeNum'] ==3 :
            return 4
    else:
        return row['Age']
traindf['Age'] = traindf.apply(pos_age, axis=1)
testdf['Age'] = testdf.apply(pos_age, axis=1)


In [ ]:
insights(traindf, 'Age', 'Survived')

Now our Age feature conveys way more information than it did earlier, thsnks to the the feature engineering and construction technique employed for this analysis.

ow that we have handled our Null values and performed Feature engineering on Age Feature, let us come back to 'Cabin' feature. While most of notebooks I observed tend to drop this feature, I feel that meaningful insights can be made from this feature. 

I believe that cabins were alotted ony to the 'Upper Class' travelers, hence their chances of survival increases as they were given the first preference.

Therefore, we can modify this into categorical data, replacing all raw cabin names with **Yes** and Null values with **No**. 

In [ ]:
def determine_cabin(row):
    if pd.isnull(row['Cabin']):
        return "No"
    else:
        return "Yes"
traindf['Cabin'] = traindf.apply(determine_cabin, axis=1)
testdf['Cabin'] = testdf.apply(determine_cabin, axis=1)

In [ ]:
insights(traindf, 'Cabin', 'Survived')

We clearly see, how majority of people not having cabin Perished, while on the contrary, the ones having Cabin did survive majoritily. We can hope this will improve this feature's covariance

Now we will try to improve and clean the data of 'Fare' Feature. clearly we can split the numeric data into different categorical ranges of ticket prices, as this also indirectly tells about the 'Class' of passengers just like Cabin did.

In [ ]:
insights(traindf, 'Fare', 'Survived')

In [ ]:
def determine_fare(row):
    if row['Fare'] <= 10:
        return 0
    elif row['Fare'] > 10 and row['Fare'] <= 16:
        return 1
    elif row['Fare'] > 16 and row['Fare'] <= 55:
        return 2
    else:
        return 3
traindf['Fare'] = traindf.apply(determine_fare, axis=1)
testdf['Fare'] = testdf.apply(determine_fare, axis=1)

In [ ]:
insights(traindf, 'Fare', 'Survived')

Here, the 'Classes' can be thought of as follow:
1. 0-> Lower Economy class
2. 1-> Economy class
3. 2-> Higher Economy class
4. 3-> Rich class

### 5. Final covariance matrix check
​
Now that we have handled Null values and Feature Engineering, it's time to have a look again at the covariance matrix of our features with our Target variable, and look for improvements in covariance.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
encoder = LabelEncoder()
scaler= StandardScaler()
tren = traindf.apply(lambda i: encoder.fit_transform(i) if i.dtype == 'O' else i)
trcov = pd.DataFrame(scaler.fit_transform(tren), columns=tren.columns)
cov_matrix = trcov.cov()
plt.figure(figsize=(12, 12))
sns.heatmap(cov_matrix, annot=True, fmt='.3f', linewidths=.75)
plt.title('Covariance')
plt.show()

We can clearly see how covariance of various Features have changed drastically with our Target variable. Some observations are as follow:-
1. **Age**(-0.078 to -0.153)
2. **Fare**(0.258 to 0.337)
3. **Cabin**(-0.255 to 0.317)

Moreover, the new column **RelativeNum** has better covariance than it's predecessors 'SibSp' and 'Parch'

Now let's drop the features that arent of much interest. Remember that since we already have a rather **small data and number of features** we **can't eliminate** many features.

We eliminated: 
1. **PassengerId and Name**: not much contribution to Covariance.
2. **Ticket**: I feel nothing much can be figured out from it's extremely raw data.
3. **SibSp and Parch**: We used them to construct a higher covariance column, thus we don't need them anymore.

In [ ]:
tren=tren.drop(['PassengerId', 'Name', 'Ticket', 'SibSp','Parch'], axis=1)

### 6. Dataset preparation for fitting in models

Now let's prepare our dataset to fit into our **Machine Learning models**. firstly, we will split our train data into **X(Independent features)** and **Y(Survived feature)** for fitting in model.

Simultaneously we are also preparing our test dataset, first by **encoding** it using the same function which we ised to encode our train data, then **removing** unnecessary features.

In [ ]:
X_train=tren.drop(['Survived'], axis=1)
Y_train=tren['Survived']
testdf = testdf.apply(lambda i: encoder.fit_transform(i) if i.dtype == 'O' else i)
testdf=testdf.drop([ 'Name', 'Ticket', 'SibSp','Parch'], axis=1)
X_test=testdf.drop(['PassengerId'], axis=1)

Next, we scale our data to Standard Normal Distribution so that all entries are on same scale, thus enhancing our model predictions further.

In [ ]:
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)

### 7. Model evaluation
​
We will fit our data into three boosting models, namely:
1. Gradient Boosting Classifier
1. Light Gradient Boosting Machine Classifier (LGBM CLassifier)
2. Extreme Gradient Boosting Classifier (XGB CLaassifier)
​
Furthermore, we will perform **K-Folds cross validation** on our models to evaluate its performance, take the average score for **k=5** and select the most suitable one for our Final Submission.

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

#### 1. Light Gradient Boosting Machine Classifier (LGBM CLassifier)

In [ ]:
modelL=LGBMClassifier()
modelL.fit(X_train, Y_train)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(modelL, X_train, Y_train, cv=kf)
print("Accuracy for each fold:", cv_results)
print("Average accuracy across all folds:", cv_results.mean())

#### 2. Extreme Gradient Boosting Classifier

In [ ]:
modelX= XGBClassifier()
modelX.fit(X_train, Y_train)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(modelX, X_train, Y_train, cv=kf)
print("Accuracy for each fold:", cv_results)
print("Average accuracy across all folds:", cv_results.mean())

#### 3. Gradient Boosting Classifier

In [ ]:
modelG= GradientBoostingClassifier(n_estimators=100, learning_rate=0.01, random_state=42)
modelG.fit(X_train, Y_train)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(modelG, X_train, Y_train, cv=kf)
print("Accuracy for each fold:", cv_results)
print("Average accuracy across all folds:", cv_results.mean())

Based on these evaluation results, I selected **Gradient Boosting Classifier** model for this task.

### 8. Final submission

As stated, I generated predictions using Gradient Boosting Classifier model, after which I made excel in specified format, and submitted this notebook for score calculation.

In [ ]:
pred=modelG.predict(X_test)
testdf['Survived']=pred
Submission=testdf[['PassengerId', 'Survived']]
Submission.to_csv('submission.csv', index=None)